In [3]:
!pip install torch torchvision timm scikit-learn pillow tqdm

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import timm

In [5]:
DATASET_PATH = r"D:\ML-PROJECTS\long-hair detection\datasets\age\UTKFace"

In [6]:
data = []

for file in os.listdir(DATASET_PATH):

    if file.endswith(".jpg"):

        try:
            age = int(file.split("_")[0])
            gender = int(file.split("_")[1])

            data.append([
                os.path.join(DATASET_PATH, file),
                gender
            ])

        except:
            pass

df = pd.DataFrame(data, columns=["image_path","gender"])

print(df.head())
print()
print("Total Images:", len(df))

                                          image_path  gender
0  D:\ML-PROJECTS\long-hair detection\datasets\ag...       0
1  D:\ML-PROJECTS\long-hair detection\datasets\ag...       0
2  D:\ML-PROJECTS\long-hair detection\datasets\ag...       1
3  D:\ML-PROJECTS\long-hair detection\datasets\ag...       1
4  D:\ML-PROJECTS\long-hair detection\datasets\ag...       1

Total Images: 23708


In [7]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["gender"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["gender"],
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 18966
Val: 2371
Test: 2371


In [8]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [9]:
class GenderDataset(Dataset):

    def __init__(self, df, transform=None):

        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        img_path = self.df.loc[idx,"image_path"]
        gender = self.df.loc[idx,"gender"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, gender

In [10]:
train_dataset = GenderDataset(
    train_df,
    train_transform
)

val_dataset = GenderDataset(
    val_df,
    test_transform
)

test_dataset = GenderDataset(
    test_df,
    test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [11]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [12]:
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=2
)

model = model.to(device)

In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [14]:
EPOCHS = 50

best_acc = 0

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    model.eval()

    preds = []
    actuals = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            outputs = model(images)

            pred = torch.argmax(outputs, dim=1)

            preds.extend(pred.cpu().numpy())
            actuals.extend(labels.numpy())

    acc = accuracy_score(actuals, preds)

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss:{running_loss:.4f} "
        f"Val Acc:{acc:.4f}"
    )

    if acc > best_acc:

        best_acc = acc

        torch.save(
            model.state_dict(),
            "gender_model.pth"
        )

        print("Saved Best Model")

 83%|████████▎ | 490/593 [03:27<00:43,  2.36it/s]


KeyboardInterrupt: 